In [1]:
from pathlib import Path
import csv
import json
import math
import re
import statistics
from collections import defaultdict

# Shared definitions intentionally included inside each notebook.
# No external Python helper file is needed.
DATA_FOLDER = Path.cwd() / "csv_data"
CONDITIONS = ["Live", "20 mg/L", "320 mg/L", "Dead"]
LABELS = {"live": "Live", "20": "20 mg/L", "320": "320 mg/L", "dead": "Dead"}
FOLDER_RE = re.compile(r"^(Live|20|320|Dead)_cell(\d+)_crop(\d+)$", re.I)
LIMIT_RMS_PN = 25.0
MIN_LINEAR_NM = 5.0
MIN_ELASTICITY_NM = 10.0
MAX_OVERLAP_NM = 5.0


def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def num(value):
    try:
        result = float(value)
        return result if math.isfinite(result) else None
    except (ValueError, TypeError, OverflowError):
        return None


def identity(row):
    return (row.get("filename", ""), row.get("curve_number", ""),
            row.get("position_index", ""))


def interval(row, prefix):
    a = num(row.get(prefix + "_x_min_used_nm"))
    b = num(row.get(prefix + "_x_max_used_nm"))
    return tuple(sorted((a, b))) if a is not None and b is not None else None


def widths(row):
    linear = interval(row, "linear")
    elastic = interval(row, "elasticity")
    linear_nm = linear[1] - linear[0] if linear else None
    elasticity_nm = elastic[1] - elastic[0] if elastic else None
    # Indentation-side part of the fitted elasticity interval, relative to
    # the processed curve's contact at X = 0 nm, clipped to the fit interval.
    contact_nm = (max(0.0, min(0.0, elastic[1]) - elastic[0])
                  if elastic else None)
    overlap_nm = (max(0.0, min(linear[1], elastic[1]) - max(linear[0], elastic[0]))
                  if linear and elastic else None)
    xc = num(row.get("contact_point_x_nm"))
    fitted_contact_nm = (max(0.0, min(max(xc, elastic[0]), elastic[1]) - elastic[0])
                         if elastic and xc is not None else None)
    return linear_nm, elasticity_nm, contact_nm, fitted_contact_nm, overlap_nm


def trace(points, y_column, used_column=None):
    selected = []
    for row in points:
        if used_column and row.get(used_column) != "1":
            continue
        x, y = num(row.get("x_nm")), num(row.get(y_column))
        if x is not None and y is not None:
            selected.append([x, y])
    if used_column:
        selected.sort(key=lambda point: point[0])
    return selected


def elasticity_rms_from_points(points):
    """Recalculate custom RMS from actual plotted fitted points, in pN.

    Force values in the point CSV are nN, so multiply RMS by 1000.
    N is the number of valid pairs in the elasticity fitting interval.
    The three fitted parameters are E, contact position and baseline.
    """
    residual_sq = []
    for point in points:
        if point.get("elasticity_fit_used") != "1":
            continue
        measured = num(point.get("measured_y_nN"))
        predicted = num(point.get("elasticity_fit_y_nN"))
        if measured is not None and predicted is not None:
            residual_sq.append((measured - predicted) ** 2)
    n = len(residual_sq)
    if n <= 3:
        return None, n
    return math.sqrt(math.fsum(residual_sq) / (n - 3)) * 1000.0, n


def derived_values(summary, points):
    if summary is None:
        return {}
    v = dict(summary)
    slope = num(v.get("linear_slope_bruker_N_per_m"))
    linear_nm, elasticity_nm, contact_nm, fitted_nm, overlap_nm = widths(v)
    v["bacterial_spring_constant_nN_per_um"] = abs(slope) * 1000 if slope is not None else None
    v["linear_fit_range_nm"] = linear_nm
    v["elasticity_fit_range_nm"] = elasticity_nm
    v["elasticity_fit_range_to_preprocessing_contact_nm"] = contact_nm
    v["elasticity_fit_range_to_fitted_contact_nm"] = fitted_nm
    v["fit_overlap_nm"] = overlap_nm
    # Recompute directly instead of multiplying the old RMS a second time.
    # Correct for both old N and newer N-3 Jython exports, including 320 Cell 3.
    v["elasticity_rmsd_custom_pN"], v["elasticity_recalculated_T"] = elasticity_rms_from_points(points)
    jpk = num(v.get("elasticity_rms_bruker_pN"))
    custom = v["elasticity_rmsd_custom_pN"]
    v["elasticity_rms_difference_pN"] = (custom - jpk if custom is not None and jpk is not None else None)
    if custom is not None and jpk is not None:
        avg = (abs(custom) + abs(jpk)) / 2
        v["elasticity_rms_percentage_difference"] = abs(custom - jpk) / avg * 100 if avg else 0.0
    else:
        v["elasticity_rms_percentage_difference"] = None
    return v


def rejection_reasons(v):
    if not v:
        return ["No uniquely matching summary row"]
    checks = [
        (num(v.get("linear_rmsd_custom_pN")), "Linear RMS > 25 pN or missing", "max", LIMIT_RMS_PN),
        (num(v.get("elasticity_rms_bruker_pN")), "JPK elasticity RMS > 25 pN or missing", "max", LIMIT_RMS_PN),
        (v.get("linear_fit_range_nm"), "Linear fitting width < 5 nm or missing", "min", MIN_LINEAR_NM),
        (v.get("elasticity_fit_range_to_preprocessing_contact_nm"), "Elasticity indentation width < 10 nm or missing", "min", MIN_ELASTICITY_NM),
        (v.get("fit_overlap_nm"), "Fitting interval overlap > 5 nm or missing", "max", MAX_OVERLAP_NM),
    ]
    reasons = []
    for value, reason, direction, limit in checks:
        if value is None or (direction == "max" and value > limit) or (direction == "min" and value < limit):
            reasons.append(reason)
    return reasons


def curve_inventory():
    """Read point CSVs first, match summaries by full filename/curve/position.

    If historic curve_number in a point file is wrong, recover by a unique
    filename + position match. Summary-only leftovers are never included.
    """
    if not DATA_FOLDER.is_dir():
        raise FileNotFoundError("Run this notebook from the repository root containing csv_data")
    folders = [p for p in DATA_FOLDER.iterdir() if p.is_dir() and FOLDER_RE.fullmatch(p.name)]
    folders.sort(key=lambda p: (CONDITIONS.index(LABELS[FOLDER_RE.fullmatch(p.name).group(1).lower()]),
                                int(FOLDER_RE.fullmatch(p.name).group(2)),
                                int(FOLDER_RE.fullmatch(p.name).group(3))))
    for folder in folders:
        matcher = FOLDER_RE.fullmatch(folder.name)
        condition = LABELS[matcher.group(1).lower()]
        cell, crop = int(matcher.group(2)), int(matcher.group(3))
        paths = sorted(folder.glob("AFM_fit_points_curve_*.csv"))
        if not paths:
            continue
        summary_file = folder / "AFM_jython_summary.csv"
        summaries = read_csv(summary_file) if summary_file.exists() else []
        by_key, by_fallback = defaultdict(list), defaultdict(list)
        for row in summaries:
            if row.get("filename") and row.get("position_index"):
                by_key[identity(row)].append(row)
                by_fallback[(row["filename"], row["position_index"])].append(row)
        counted = set()
        for path in paths:
            points = read_csv(path)
            if not points:
                continue
            first = points[0]
            matches = by_key.get(identity(first), [])
            if not matches:
                matches = by_fallback.get((first.get("filename", ""), first.get("position_index", "")), [])
            summary = matches[0] if len(matches) == 1 else None
            key = identity(summary) if summary else None
            duplicate = key in counted if key else False
            if key:
                counted.add(key)
            values = derived_values(summary, points)
            reasons = rejection_reasons(values)
            if duplicate:
                reasons.append("Duplicate individual fit-point export")
            measured = trace(points, "measured_y_nN")
            linear = trace(points, "linear_fit_y_nN", "linear_fit_used")
            elastic = trace(points, "elasticity_fit_y_nN", "elasticity_fit_used")
            if not (measured and linear and elastic):
                reasons.append("Missing measured or fitted plotting points")
            yield {
                "condition": condition, "cell": cell, "crop": crop, "folder": folder.name,
                "curve": (summary.get("curve_number") if summary else first.get("curve_number")),
                "position": first.get("position_index"), "filename": first.get("filename", ""),
                "values": values, "reasons": reasons, "accepted": not reasons,
                "measured": measured, "linear": linear, "elasticity": elastic,
            }


def med(values):
    return statistics.median(values) if values else None


def mad(values):
    if not values:
        return None
    centre = statistics.median(values)
    return statistics.median(abs(v - centre) for v in values)


def grouping(curves):
    groups = defaultdict(list)
    for curve in curves:
        if curve["accepted"]:
            groups[(curve["condition"], curve["cell"])].append(curve)
    return groups


def cell_statistics(curves):
    result = {}
    for key, group in grouping(curves).items():
        kb = [num(c["values"].get("bacterial_spring_constant_nN_per_um")) for c in group]
        young = [num(c["values"].get("youngs_modulus_MPa")) for c in group]
        kb, young = [x for x in kb if x is not None], [x for x in young if x is not None]
        result[key] = {
            "n": len(group), "kb_median": med(kb), "kb_mad": mad(kb),
            "E_median": med(young), "E_mad": mad(young)
        }
    return result


def condition_statistics(cells):
    result = {}
    for cond in CONDITIONS:
        rows = [v for (c, _), v in sorted(cells.items(), key=lambda t: t[0][1]) if c == cond]
        def aggregate(key):
            v = [r[key] for r in rows if r[key] is not None]
            return (statistics.mean(v) if v else None,
                    statistics.stdev(v) if len(v) >= 2 else None)
        result[cond] = (aggregate("kb_median"), aggregate("E_median"))
    return result

from html import escape

curves = list(curve_inventory())
cells = cell_statistics(curves)
condition_stats = condition_statistics(cells)

# Cell medians/MADs; condition means and sample SDs of the three cell medians.
def fmt(value, decimals=4):
    return "" if value is None else f"{value:.{decimals}f}"

summary_rows = []
for condition in CONDITIONS:
    keys = sorted((key for key in cells if key[0] == condition), key=lambda key: key[1])
    k_aggregate, e_aggregate = condition_stats[condition]
    for ix, key in enumerate(keys):
        stat = cells[key]
        summary_rows.append("<tr>" +
            (f'<td rowspan="{len(keys)}"><b>{escape(condition)}</b></td>' if ix == 0 else "") +
            f'<td>Cell {key[1]}</td><td>{stat["n"]}</td>' +
            f'<td>{fmt(stat["kb_median"])}</td><td>{fmt(stat["kb_mad"])}</td>' +
            (f'<td rowspan="{len(keys)}">{fmt(k_aggregate[0],2)}</td><td rowspan="{len(keys)}">{fmt(k_aggregate[1],2)}</td>' if ix == 0 else "") +
            f'<td>{fmt(stat["E_median"])}</td><td>{fmt(stat["E_mad"])}</td>' +
            (f'<td rowspan="{len(keys)}">{fmt(e_aggregate[0],2)}</td><td rowspan="{len(keys)}">{fmt(e_aggregate[1],2)}</td>' if ix == 0 else "") + '</tr>')

summary_html = """<section class="card summary"><h2>AFM summary: all five acceptance rules</h2>
<table><thead><tr><th rowspan="2">Condition</th><th rowspan="2">Cell</th>
<th rowspan="2">Accepted n</th><th colspan="4">Bacterial spring constant (nN/µm)</th>
<th colspan="4">Young's modulus (MPa)</th></tr>
<tr><th>Median</th><th>MAD</th><th>Mean of medians</th><th>SD</th>
<th>Median</th><th>MAD</th><th>Mean of medians</th><th>SD</th></tr>
</thead><tbody>""" + "\n".join(summary_rows) + """</tbody></table>
<p>Limits: linear and JPK elasticity RMS ≤ 25 pN, linear width ≥ 5 nm,
elasticity indentation width to X = 0 ≥ 10 nm, overlap ≤ 5 nm.</p>
<p>Only accepted curves enter statistics. All exported curves remain selectable for inspection.</p></section>"""

datasets = []
by_folder = defaultdict(list)
for c in curves:
    # Individual point CSVs (including any rejected fit) remain viewable.
    by_folder[c["folder"]].append(c)
for folder in sorted(by_folder, key=lambda name:(
    CONDITIONS.index(LABELS[FOLDER_RE.fullmatch(name).group(1).lower()]),
    int(FOLDER_RE.fullmatch(name).group(2)), int(FOLDER_RE.fullmatch(name).group(3)))):
    items = sorted(by_folder[folder], key=lambda c: (int(c["curve"]),int(c["position"])))
    datasets.append({"name":folder,"curves":items})

fields = [('Linear slope', 'linear_slope_bruker_N_per_m', 'N/m'), ('Bacterial spring constant', 'bacterial_spring_constant_nN_per_um', 'nN/µm'), ('Linear RMSD custom', 'linear_rmsd_custom_pN', 'pN'), ('Linear X minimum', 'linear_x_min_used_nm', 'nm'), ('Linear X maximum', 'linear_x_max_used_nm', 'nm'), ('Linear fit range', 'linear_fit_range_nm', 'nm'), ("Young's modulus", 'youngs_modulus_MPa', 'MPa'), ('JPK Elasticity RMS', 'elasticity_rms_bruker_pN', 'pN'), ('Elasticity RMSD custom', 'elasticity_rmsd_custom_pN', 'pN'), ('Elasticity RMS percentage difference', 'elasticity_rms_percentage_difference', '%'), ('Elasticity X minimum', 'elasticity_x_min_used_nm', 'nm'), ('Elasticity X maximum', 'elasticity_x_max_used_nm', 'nm'), ('Elasticity fit range', 'elasticity_fit_range_nm', 'nm'), ('Elasticity fit range to preprocessing contact', 'elasticity_fit_range_to_preprocessing_contact_nm', 'nm'), ('Elasticity fit range to fitted contact', 'elasticity_fit_range_to_fitted_contact_nm', 'nm')]
html_output = '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>AFM Curve Dashboard</title>\n<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>\n<style>\n*{box-sizing:border-box}body{margin:0;background:#f4f5f7;color:#202124;font-family:Arial,sans-serif}\nheader{padding:20px 28px;background:white;border-bottom:1px solid #ddd}h1{font-size:24px;margin:0 0 5px}\nheader p{margin:0;color:#666;font-size:14px}.controls,.card{background:white;padding:20px;border-radius:8px}\n.controls{display:flex;flex-wrap:wrap;align-items:center;gap:12px;margin:20px}\nselect,button{padding:9px 12px;font-size:14px}select{max-width:100%;min-width:190px}\n.layout{display:grid;grid-template-columns:minmax(0,1fr) 390px;gap:20px;margin:20px}\n#plot{height:600px}.card{min-width:0}h2{font-size:18px;margin:0 0 14px}\n#filename,#status,#rmsnotice{font-size:13px;color:#555;line-height:1.6;overflow-wrap:anywhere}\n#rmsnotice{color:#9b4f00}.summary{margin:20px;overflow-x:auto}\ntable{border-collapse:collapse;width:100%;font-size:13px}th,td{border-bottom:1px solid #e5e8ec;padding:8px 7px;text-align:right}\nth{background:#f5f6f7}th:first-child,td:first-child{text-align:left}\n#values td:first-child{width:67%}.summary table th,.summary table td{border:1px solid #dae0e5;padding:9px}\n.accepted{color:#17643b;font-weight:bold}.rejected{color:#a6322d;font-weight:bold}\n@media(max-width:950px){.layout{grid-template-columns:1fr}#plot{height:440px}}\n</style></head>\n<body>\n<header><h1>AFM force-curve dashboard</h1><p>Measured force, linear fit, elasticity fit, and five-rule accepted-curve statistics.</p></header>\n__SUMMARY__\n<div class="controls">\n<label>Crop <select id="folders"></select></label>\n<label>Curve <select id="curves"></select></label>\n<button id="prev">Previous</button><button id="next">Next</button>\n<span id="status"></span>\n</div>\n<div class="layout"><div class="card"><div id="plot"></div><div id="filename"></div></div>\n<div class="card"><h2>Curve values</h2><p id="acceptance"></p><p id="rmsnotice"></p>\n<table id="values"></table></div></div>\n<script>\nconst datasets = __DATA__;\nconst fields = __FIELDS__;\nconst folderSelect = document.getElementById(\'folders\');\nconst curveSelect = document.getElementById(\'curves\');\nfunction numberText(v){if(v===null||v===undefined||v===\'\')return \'N/A\';const n=Number(v);return Number.isFinite(n)?String(Number(n.toPrecision(6))):\'N/A\';}\nfunction pointsTrace(points,name,color,mode,width){return {type:\'scatter\',mode,name,x:points.map(p=>p[0]),y:points.map(p=>p[1]),line:{color,width},marker:{color,size:3}};}\nfunction showCurve(){\n const ds=datasets[Number(folderSelect.value)], c=ds.curves[Number(curveSelect.value)];\n if(!c)return;\n const traces=[pointsTrace(c.measured,\'Measured\',\'#58758a\',\'lines+markers\',1.5)];\n if(c.linear.length)traces.push(pointsTrace(c.linear,\'Linear fit\',\'#2ca02c\',\'lines\',3));\n if(c.elasticity.length)traces.push(pointsTrace(c.elasticity,\'Elasticity fit\',\'#d62728\',\'lines\',3));\n const cx=Number(c.values.contact_point_x_nm),cy=Number(c.values.contact_point_y_fit_nN);\n if(c.values.contact_point_x_nm!==undefined && c.values.contact_point_y_fit_nN!==undefined&&Number.isFinite(cx)&&Number.isFinite(cy)){\n   traces.push({x:[cx],y:[cy],mode:\'markers\',type:\'scatter\',name:\'Fitted contact point\',marker:{color:\'#d62728\',size:10}});\n }\n Plotly.react(\'plot\',traces,{title:ds.name+\' | Curve \'+c.curve,xaxis:{title:\'Vertical Tip Position (nm)\',autorange:true},yaxis:{title:\'Force (nN)\',autorange:true},legend:{orientation:\'h\',y:-0.2},margin:{l:70,r:25,t:60,b:90},hovermode:\'closest\'},\n {responsive:true,displaylogo:false,toImageButtonOptions:{filename:\'AFM_curve_\'+c.curve,scale:2}});\n document.getElementById(\'status\').textContent=\'Curve \'+(Number(curveSelect.value)+1)+\' of \'+ds.curves.length+\' | Position \'+c.position;\n const accept=document.getElementById(\'acceptance\');accept.className=c.accepted?\'accepted\':\'rejected\';\n accept.textContent=c.accepted?\'Accepted: all five criteria pass\':\'Not accepted: \'+c.reasons.join(\'; \');\n const delta=c.values.elasticity_rms_difference_pN;\n document.getElementById(\'rmsnotice\').textContent=delta!==null&&delta!==undefined&&Math.abs(delta)>0.01?\n  \'Check fit interval: custom RMS differs from JPK by \'+numberText(delta)+\' pN. The custom value is independently recalculated.\':\'\';\n document.getElementById(\'filename\').textContent=c.filename;\n const table=document.getElementById(\'values\');table.replaceChildren();\n for(const [label,key,unit] of fields){const tr=document.createElement(\'tr\');\n for(const value of [label,numberText(c.values[key]),unit]){const td=document.createElement(\'td\');td.textContent=value;tr.appendChild(td)}table.appendChild(tr)}\n document.getElementById(\'prev\').disabled=curveSelect.selectedIndex===0;\n document.getElementById(\'next\').disabled=curveSelect.selectedIndex>=curveSelect.length-1;\n}\nfunction showFolder(){const ds=datasets[Number(folderSelect.value)];curveSelect.replaceChildren();\n ds.curves.forEach((c,i)=>curveSelect.add(new Option(\'Curve \'+c.curve+\' | Position \'+c.position,i)));\n if(ds.curves.length)showCurve();}\ndatasets.forEach((ds,i)=>folderSelect.add(new Option(ds.name,i)));\nfolderSelect.onchange=showFolder;curveSelect.onchange=showCurve;\ndocument.getElementById(\'prev\').onclick=()=>{if(curveSelect.selectedIndex>0){curveSelect.selectedIndex--;showCurve()}};\ndocument.getElementById(\'next\').onclick=()=>{if(curveSelect.selectedIndex<curveSelect.length-1){curveSelect.selectedIndex++;showCurve()}};\nif(datasets.length)showFolder();\n</script></body></html>\n'
html_output = (html_output.replace('__SUMMARY__', summary_html)
    .replace('__DATA__',json.dumps(datasets,ensure_ascii=False,allow_nan=False).replace('</','<\\/'))
    .replace('__FIELDS__',json.dumps(fields,ensure_ascii=False)))
output = Path.cwd() / 'index.html'
output.write_text(html_output,encoding='utf-8')

accepted = sum(c['accepted'] for c in curves)
print('Created:',output)
print('Curve CSVs:',len(curves),'| Accepted by all five rules:',accepted,
      '| Rejected/unmatched:',len(curves)-accepted)
for condition in CONDITIONS:
    print(condition,':',[(cell,cells.get((condition,cell),{}).get('n',0)) for cell in (1,2,3)])
for c in curves:
    delta = c['values'].get('elasticity_rms_difference_pN')
    if delta is not None and abs(delta) > 0.01:
        print('RMS interval discrepancy:',c['folder'],'Curve',c['curve'],
              'custom minus JPK =',round(delta,6),'pN')


Created: /workspaces/AFM-plots_Jython_exports/index.html
Curve CSVs: 615 | Accepted by all five rules: 615 | Rejected/unmatched: 0
Live : [(1, 50), (2, 52), (3, 51)]
20 mg/L : [(1, 50), (2, 53), (3, 50)]
320 mg/L : [(1, 50), (2, 51), (3, 53)]
Dead : [(1, 53), (2, 51), (3, 51)]
RMS interval discrepancy: Dead_cell1_crop2 Curve 126 custom minus JPK = -0.529936 pN
RMS interval discrepancy: Dead_cell2_crop1 Curve 181 custom minus JPK = -0.595146 pN
RMS interval discrepancy: Dead_cell2_crop1 Curve 99 custom minus JPK = -0.723249 pN
RMS interval discrepancy: Dead_cell3_crop1 Curve 5 custom minus JPK = -0.053981 pN
